# University of Kentucky Flowsheet Optimization Tutorial 
Throughout this tutorial, you will learn how to **write an optimization function** for a given flowsheet, **iteratively optimize the flowsheet**, and **visualize the corresponding improvements** while also gaining an understanding of the engineering insights used to justify the system's operating conditions. Our use case will be the University of Kentucky (UKy) flowsheet, a continuous, pilot-scale plant operation for recovering rare earth elements (REEs) from coal and coal byproducts with the goal of producing marketable mixed rare earth oxides (REOs), as shown in the process flow diagram below. For more information on this flowsheet, please refer to the Useful Links at the bottom of this cell. 


## UKy Flowsheet Connectivity
![uky_flowsheet.png](./uky_flowsheet.png)


## Tutorial Outline
In this tutorial, we will start with an outdated version of the UKy flowsheet that is only able to achieve 0.94% REE recovery. Then, we will write an optimization function that unfixes a variety of operating conditions in order to maximize the REE recovery while still adhering to realistic variable bounds. From there, we will walk through an iterative process of running the optimization, updating the flowsheet to match the optimized conditions, diagnosing any issues, expanding the bounds of the optimization, and then repeating the process until we hit the target recovery of ~20% REE.


### Useful Links:
- Public Github Repository: https://github.com/prommis/prommis/tree/main
- Current UKy Flowsheet Code: https://github.com/prommis/prommis/blob/main/src/prommis/uky/uky_flowsheet.py
- Outdated UKy Flowsheet Code: [Outdated UKy](uky_flowsheet_low_recovery.py)
- Tutorial on Building the UKy Flowsheet: [UKy Flowsheet Tutorial](uky_flowsheet-solution.ipynb)


## Step 1: Import the necessary tools

We'll use some basic functionalities from Pyomo, generic models from IDAES, and process-specific models from PrOMMiS.

In [1]:
# Import the essentials from Pyomo
from pyomo.environ import (
    assert_optimal_termination,
    ConcreteModel,
    Constraint,
    Expression,
    Objective,
    Param,
    SolverFactory,
    Suffix,
    TransformationFactory,
    Var,
    check_optimal_termination,
    units,
    value,
)
from pyomo.network import Arc, SequentialDecomposition

# Import the essentials from IDAES
import idaes.logger as idaeslog
from idaes.core import (
    FlowDirection,
    FlowsheetBlock,
    MaterialBalanceType,
    MomentumBalanceType,
    UnitModelBlock,
    UnitModelCostingBlock,
)

# Import scaling, initialization, and diagnostic tools from IDAES
from idaes.core.initialization import BlockTriangularizationInitializer
from idaes.core.scaling.scaling_base import ScalerBase
from idaes.core.scaling import CustomScalerBase, ConstraintScalingScheme
from idaes.core.scaling.util import get_scaling_factor, set_scaling_factor
from idaes.core.solvers import get_solver
from idaes.core.util.model_diagnostics import DiagnosticsToolbox
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.models.properties.modular_properties.base.generic_property import (
    GenericParameterBlock,
)

# Import unit models from IDAES
from idaes.models.unit_models.feed import Feed, FeedInitializer
from idaes.models.unit_models.mixer import (
    Mixer,
    MixerInitializer,
    MixingType,
    MomentumMixingType,
)
from idaes.models.unit_models.product import Product, ProductInitializer
from idaes.models.unit_models.separator import (
    EnergySplittingType,
    Separator,
    SeparatorInitializer,
    SplittingType,
)
from idaes.models.unit_models.solid_liquid import SLSeparator
from idaes.models_extra.power_generation.properties.natural_gas_PR import (
    EosType,
    get_prop,
)

# Import the UKy-specific unit and property models
from prommis.leaching.leach_reactions import CoalRefuseLeachingReactionParameterBlock
from prommis.properties.coal_refuse_properties import CoalRefuseParameters
from prommis.properties.sulfuric_acid_leaching_properties import (
    SulfuricAcidLeachingParameters,
)
from prommis.leaching.leach_train import LeachingTrain, LeachingTrainInitializer
from prommis.precipitate.precipitate_liquid_properties import AqueousParameter
from prommis.precipitate.precipitate_solids_properties import PrecipitateParameters
from prommis.precipitate.precipitator import Precipitator
from prommis.roasting.ree_oxalate_roaster import REEOxalateRoaster
from prommis.solvent_extraction.ree_og_distribution import REESolExOgParameters
from prommis.solvent_extraction.solvent_extraction import (
    SolventExtraction,
    SolventExtractionInitializer,
)
from prommis.solvent_extraction.translator_leach_precip import TranslatorLeachPrecip
from prommis.solvent_extraction.solvent_extraction_reaction_package import (
    SolventExtractionReactions,
)
from prommis.uky.costing.costing_dictionaries import load_REE_costing_dictionary
from prommis.uky.costing.ree_plant_capcost import QGESSCosting, QGESSCostingData

# Import the outdated UKy flowsheet
import uky_flowsheet_low_recovery as uky

# Set up logger
_log = idaeslog.getLogger(__name__)

## Step 2: Run the outdated UKy Flowsheet
[Add explanation about how we're just importing functions from the outdated flowsheet. Also explain that capture is used to suppress output]

In [2]:
%%capture
m, _ = uky.main()
# m = uky.build()
# uky.set_operating_conditions(m)
# uky.set_scaling(m)
# uky.initialize_system(m)
# uky.solve_system(m)
# uky.fix_organic_recycle(m)
# uky.add_result_expressions(m)
# uky.display_results(m)

2026-06-08 16:21:04 [INFO] idaes.uky_flowsheet_low_recovery: Initialization Order: {_init_ord}
2026-06-08 16:21:04 [INFO] idaes.uky_flowsheet_low_recovery: Initializing fs.leach_solid_feed
2026-06-08 16:21:04 [INFO] idaes.uky_flowsheet_low_recovery: Initializing fs.leach_liquid_feed
2026-06-08 16:21:04 [INFO] idaes.uky_flowsheet_low_recovery: Initializing fs.solex_rougher_load
2026-06-08 16:21:05 [INFO] idaes.init.fs.solex_rougher_load.mscontactor: Stream Initialization Completed.
2026-06-08 16:21:05 [INFO] idaes.init.fs.solex_rougher_load.mscontactor: Initialization Completed, optimal - <undefined>
2026-06-08 16:21:05 [INFO] idaes.uky_flowsheet_low_recovery: Initializing fs.rougher_org_make_up
2026-06-08 16:21:06 [INFO] idaes.uky_flowsheet_low_recovery: Initializing fs.acid_feed1
2026-06-08 16:21:06 [INFO] idaes.uky_flowsheet_low_recovery: Initializing fs.acid_feed2
2026-06-08 16:21:06 [INFO] idaes.uky_flowsheet_low_recovery: Initializing fs.solex_cleaner_load
2026-06-08 16:21:06 [INF

In [3]:
print(f"Total REE recovery is {value(m.fs.overall_ree_recovery_percentage[0])} %")

Total REE recovery is 0.904960020448646 %


## Step 3: Write an optimization function to maximize REE recovery
May want to find a way to split this up, probably by embedding smaller functions inside of this one (Ex: def unfix_leach_liquid_feed). Should also show how adding each step affects recovery?

In [4]:
def optimize_model(m):
    # Goal: Minimize this objective function
    m.obj = Objective(
        expr=(
            # Assign a small penalty for having larger flow rates than necessary
            0.01
            * (
                m.fs.leach_liquid_feed.flow_vol[0]
                + m.fs.acid_feed1.flow_vol[0]
                + m.fs.acid_feed2.flow_vol[0]
                + m.fs.acid_feed3.flow_vol[0]
            )
            # Assign a large penalty for having contaminants
            + 1e5
            * (
                + m.fs.metal_product_flow[0, "Al"]
                + m.fs.metal_product_flow[0, "Ca"]
                + m.fs.metal_product_flow[0, "Fe"]
            )
            # Assign a large incentive for increasing REE product flow (REE recovery)
            - 1e5 * m.fs.ree_product_flow[0]
        )
    )

    # Now we must give this function the ability to manipulate certain variables to achieve the objective defined above
    # We will do this by unfixing certain variables and applying realistic bounds

    # Unfix the H2SO4 feed rate and feed concentration
    m.fs.leach_liquid_feed.flow_vol.unfix()

    m.fs.leach_liquid_feed.conc_mass_comp[0, "H"].unfix()
    m.fs.leach_liquid_feed.conc_mass_comp[0, "HSO4"].unfix()
    m.fs.leach_liquid_feed.conc_mass_comp[0, "SO4"].unfix()

    # Ensure that stoichiometry is upheld
    @m.fs.leach_liquid_feed.Constraint(m.fs.time)
    def H2SO4_stoich_eqn(b, t):
        return (
            b.properties[t].conc_mol_comp["H"]
            == 2 * b.properties[t].conc_mol_comp["SO4"]
            + b.properties[t].conc_mol_comp["HSO4"]
        )
    # Apply a scaling factor of 10 to conc_mol_comp values - should double check if this is necessary
    for condata in m.fs.leach_liquid_feed.H2SO4_stoich_eqn.values():
        set_scaling_factor(condata, 10)

    # Ensure chemical equilibrium for HSO4 dissociation
    @m.fs.leach_liquid_feed.Constraint(m.fs.time)
    def HSO4_dissociation(b, t):
        return (
            b.properties[t].params.Ka2 * b.properties[t].conc_mol_comp["HSO4"]
            == b.properties[t].conc_mol_comp["SO4"] * b.properties[t].conc_mol_comp["H"]
        )
    # Update scaling factors
    sf = get_scaling_factor(m.fs.leach.mscontactor.liquid[0, 1].hso4_dissociation)
    for condata in m.fs.leach_liquid_feed.HSO4_dissociation.values():
        set_scaling_factor(condata, sf)

    # We should think about how strong of acid we can use for leaching
    m.fs.leach_liquid_feed.properties[0].pH_phase["liquid"].setlb(0)

    # IPOPT wanted to use a liquid feed of 13.7 L/hr, which would be
    # far too low to let the leach solids move as a slurry.
    # The original flow rate is 224.3 L/hr, so use 100 as a rough
    # estimate for a lower bound.
    m.fs.leach_liquid_feed.properties[0].flow_vol.setlb(100)

    m.fs.rougher_mixer.outlet.flow_vol.unfix()
    m.fs.cleaner_mixer.outlet.flow_vol.unfix()

    # Unfix HCl feed flow rates and concentrations
    for feed in [m.fs.acid_feed1, m.fs.acid_feed2, m.fs.acid_feed3]:
        # Trying to optimize acid_feed1 will bring its flow to zero, so we'll keep it fixed
        if feed == m.fs.acid_feed1:
            continue
        else:
            feed.flow_vol.unfix()
        # feed.flow_vol.unfix()
        feed.conc_mass_comp[0, "H"].unfix()
        feed.conc_mass_comp[0, "Cl"].unfix()
        # Revisit how strong of an acid we can use
        feed.properties[0].pH_phase["liquid"].setlb(0)

        @feed.Constraint(m.fs.time)
        def HCl_stoich_eqn(b, t):
            return (
                b.properties[t].conc_mol_comp["H"]
                == b.properties[t].conc_mol_comp["Cl"]
            )

        for condata in feed.HCl_stoich_eqn.values():
            set_scaling_factor(condata, 10)

    m.fs.rougher_org_make_up.conc_mass_comp[0, "DEHPA"].unfix()
    m.fs.cleaner_org_make_up.conc_mass_comp[0, "DEHPA"].unfix()
    m.fs.rougher_org_make_up.properties[0].extractant_dosage.bounds = (5, 20)
    m.fs.cleaner_org_make_up.properties[0].extractant_dosage.bounds = (5, 20)

    # If the pH in the rougher scrub goes above 5 or 6, the equations get
    # extremely ill conditioned.
    m.fs.solex_rougher_scrub.mscontactor.aqueous[0.0, 1].pH_phase["liquid"].setub(6)

In [5]:
optimize_model(m)
# uky.solve_system(m, tee=True)
solver = get_solver("ipopt_v2")
solver.options.constr_viol_tol = 1e-8
solver.options.max_iter = 300

results = solver.solve(m, tee=True)
# Uncomment to display various system metrics like component recovery or leaching recovery
# uky.display_results(m)

Ipopt 3.13.2: linear_solver="ma57"
max_iter=300
nlp_scaling_method="gradient-based"
tol=1e-06
constr_viol_tol=1e-08


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:


In [6]:
print(f"Total REE recovery is {value(m.fs.overall_ree_recovery_percentage[0])} %")

Total REE recovery is 22.184783577097868 %


Talk about how our DEHPA dosage is currently bounded between 5 and 20%, but we want it to get to 25%. If we just increase the upper bound in the previous optimize function to 25% and rerun the notebook up to this point, you'll notice that the system only "Solved to Acceptable Level". In order to get more confidence in our results we'll need to...

## Step 4: Update the base conditions of the flowsheet
We'll want to match the specs of the optimized system above, but we'll also need to update the initialization routine, which is currently hard-coding initial guesses based on the unoptimized system

### Step 4.1: Print the optimized conditions
First, let's display the values for the condition optimized by the optimization function

In [7]:
m.fs.leach_liquid_feed.display()
m.fs.rougher_org_make_up.display()
m.fs.cleaner_org_make_up.display()
m.fs.acid_feed1.display()
m.fs.acid_feed2.display()
m.fs.acid_feed3.display()

Block fs.leach_liquid_feed

  Variables:
    flow_vol : Size=1, Index=fs._time, ReferenceTo=fs.leach_liquid_feed.properties[:].component('flow_vol')
        Key : Lower : Value              : Upper : Fixed : Stale : Domain
        0.0 :   100 : 100.00000068938957 :  None : False : False :  Reals
    conc_mass_comp : Size=17, Index=fs._time*fs.leach_soln.component_list, ReferenceTo=fs.leach_liquid_feed.properties[:].component('conc_mass_comp')[...]
        Key           : Lower : Value              : Upper : Fixed : Stale : Domain
          (0.0, 'Al') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Ca') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Ce') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Cl') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Dy') : 1e-20 :              1e-10 :  None :  True :  True :  Reals
          (0.0, 'Fe') : 1e-20 :             

### Step 4.2: Update the operating conditions
We will need to update XYZ based on the results of the optimization

In [8]:
m.fs.leach_liquid_feed.flow_vol.fix(100)
m.fs.leach_liquid_feed.conc_mass_comp[0, "H"].fix(277.1)
m.fs.leach_liquid_feed.conc_mass_comp[0, "HSO4"].fix(25030.2)
m.fs.leach_liquid_feed.conc_mass_comp[0, "SO4"].fix(914.8)
# ...

Try to solve for 25% after updating the operating conditions, but still won't work
Update tear guesses based on optimization solutions
After updating, run and confirm that you get the same recovery as the optimized solution, now you can expand the horizons of the optimization function
Follow the same rules of updating the operating conditions to reproduce the results
Show sensitivity plots/other visualizations of results
Play around with recycle split fractions b4 or after the visualization